# Caderno de análise — PQC no ATmega4809 (CMP223)

**Autor:** Eduardo Silva Abdala (INF/UFRGS, 2026)

Este caderno reúne toda a análise dos experimentos que executei na bancada com o Arduino Nano Every (ATmega4809 @ 20 MHz, 6 KB de SRAM e 48 KB de Flash). Todo o processamento parte diretamente dos arquivos brutos em `data/experiments/` (`raw/telemetry_*.json`, `raw/la_*.csv` e `bin/*.hex`), sem depender de tabelas intermediárias pré-geradas, e reconstrói os CSVs em `results/processed/`, as tabelas LaTeX em `results/tables/` e todas as figuras do artigo (`results/figures/`) e dos slides (`slides/figures/`).

**Resumo do que já foi executado e resolvido até esta etapa parcial:**
1. **Evolução das campanhas (17 baterias de bancada, 10.124 observações):** comecei validando apenas o Kyber-512 via serial em 04/11, adicionei o CTRU-Light, a flag `-Os` e o baseline clássico ECDH (`micro-ecc`) em 05/11, padronizei o ambiente de compilação via Nix (`avr-gcc 13.2.0`) em 07–08/11 e fechei a campanha fatorial completa com o analisador lógico de 24 MHz acoplado ao pino `PA0` (`D2`).
2. **Isolamento do atraso serial:** nas primeiras campanhas a latência era medida no host (`t_exec_ms`, do envio do comando até o 1º byte de resposta na UART). Ao medir o pulso físico no pino `PA0` (`t_la_ms`), separei o tempo puro de CPU do custo de serialização da chave/cifra (+22 a +31 ms).
3. **Ajuste de Flash ROM:** o Kyber-512 `ref` original (~79 KB) e o CTRU-Light `ref -O3` estouravam os 48 KB de Flash do ATmega4809. Refatorei constantes para `PROGMEM` e ativei `-mrelax -mcall-prologues` no `Makefile`, fazendo todas as 12 combinações caberem na placa.
4. **O que ficou para a etapa final:** a medição física de energia (integrando a queda no resistor *shunt* durante o pulso do pino `PA0`) e a coleta nos clocks reduzidos de 10 MHz e 5 MHz.

In [ ]:
from IPython.display import Image as IPImage, display
import csv, glob, json, math, os, re, subprocess
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import plotnine as p9
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from PIL import Image

# Localiza a raiz do pacote subindo pastas até encontrar data/experiments/
ROOT = Path.cwd()
while not (ROOT / 'data' / 'experiments').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EXP = ROOT / 'data' / 'experiments'
RES = ROOT / 'results'
OUT = RES / 'processed'
FIG = RES / 'figures'
TAB = RES / 'tables'
SLIDE_FIG = ROOT / 'slides' / 'figures'
for d in (OUT, FIG, TAB, SLIDE_FIG):
    d.mkdir(parents=True, exist_ok=True)

from matplotlib import font_manager
TEXTW = 395.8225 / 72.27
for f in ('LibertinusSerif-Regular.otf', 'LibertinusSerif-Bold.otf', 'LibertinusSerif-Italic.otf'):
    try:
        path = subprocess.run(['kpsewhich', f], capture_output=True, text=True).stdout.strip()
        if path:
            font_manager.fontManager.addfont(path)
    except FileNotFoundError:
        pass
sns.set_theme(style='ticks', context='paper')
plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Libertinus Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'dejavuserif',
    'font.size': 8, 'axes.labelsize': 8, 'axes.titlesize': 8, 'legend.fontsize': 7,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'axes.linewidth': 0.6,
    'grid.linewidth': 0.4, 'lines.linewidth': 1.0, 'axes.grid': True, 'grid.alpha': 0.4,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02, 'figure.dpi': 150,
})
LABEL = {'kyber512': 'Kyber-512', 'ctru-light': 'CTRU-Light', 'ecdh': 'ECDH'}
PAL = {'kyber512': '#0072B2', 'ctru-light': '#D55E00', 'ecdh': '#009E73'}
OPS = {'K': 'keygen', 'E': 'encaps', 'D': 'decaps', 'M': 'msgcrypt'}
def br(x, nd=1):
    return f'{x:,.{nd}f}'.replace(',', 'X').replace('.', ',').replace('X', '.')

## 1. Histórico das campanhas executadas na bancada (04/11 a 08/11)

O script `src/analysis/catalog.py` inspeciona `data/experiments/` e classifica como `bench` apenas as pastas que possuem arquivos `raw/telemetry_*.json` gravados pelo `trigger.py` conversando com o ATmega4809 real. Descartei execuções sintéticas (`sim`) usadas apenas para testar o pipeline offline.

Ao todo foram **17 campanhas reais de bancada** (somando **10.124 observações**), divididas em quatro etapas conforme fui resolvendo os problemas de compilação e instrumentação:
- **Campanhas #01 a #06 (04/11):** primeiros testes no hardware apenas com `kyber512` (`asm` e `ref` em `-O3`), validando a comunicação UART a 115.200 bps e a marcação de pilha (`0xAA`).
- **Campanhas #07 a #10 (05/11):** entrada do `ctru-light`, da otimização `-Os` e, a partir da #09/#10, do baseline clássico `ecdh` (`micro-ecc`), fechando as 12 combinações fatoriais (`3 algoritmos × 2 variantes × 2 flags`). Como o `ecdh ref` leva ~14 s no `encaps`, precisei elevar o timeout serial do `trigger.py` para 35 s.
- **Campanhas #11 a #15 (07/11 e 08/11 manhã):** baterias longas ($N=30$ e $N=50$) e migração para o ambiente fixo do `flake.nix` (`avr-gcc 13.2.0`). Na campanha #12 (`exp_20261107_181223`) conectei o analisador lógico pela primeira vez nas 6 combinações `-O3`, mas o filtro inicial descartava pulsos menores que 1 ms — o que ignorava o pulso curto do `msgcrypt` (~20 µs).
- **Campanhas #16 e #17 (08/11 tarde):** ajustei o limiar de pulso do analisador lógico para `0,01 ms` no `trigger.py`, validei na campanha rápida #16 e executei a **campanha fatorial completa #17 (`exp_20261108_155653_bench_campanha_completa_la_nix_8289`)** com $N=30$ repetições nas 12 combinações (1.440 observações), salvando tanto os 12 arquivos brutos `raw/la_*.csv` (amostrados a 100 kHz) quanto a largura exata de cada pulso (`t_la_ms`) dentro do `telemetry_*.json`.

In [ ]:
cat = pd.read_csv(EXP / 'catalog.csv')
bench_exps = cat.loc[cat.source == 'bench', 'experiment'].tolist()
PAT = re.compile(r'telemetry_(?P<alg>kyber512|ctru-light|ecdh)_(?P<variant>asm|ref)(?:_(?P<opt>O3|Os)_(?P<clk>[\d.]+)MHz)?\.json')

all_rows, hist_summary = [], []
for idx, e in enumerate(bench_exps, start=1):
    files = sorted(glob.glob(str(EXP / e / 'raw' / 'telemetry_*.json')))
    la_files = sorted(glob.glob(str(EXP / e / 'raw' / 'la_*.csv')))
    algs_e, opts_e, n_e, has_t_la = set(), set(), 0, False
    for f in files:
        m = PAT.match(os.path.basename(f)).groupdict()
        algs_e.add(m['alg'])
        opt = '-' + (m['opt'] or 'O3')
        opts_e.add(opt)
        data = json.load(open(f))
        n_e += len(data)
        for i, r in enumerate(data):
            t_exec = float(r['t_exec_ms'])
            t_la = float(r['t_la_ms']) if r.get('t_la_ms') is not None else np.nan
            if not np.isnan(t_la): has_t_la = True
            all_rows.append({
                **r, 'experiment': e, 'campaign_idx': idx, 'seq': i,
                'algorithm': m['alg'], 'variant': m['variant'],
                'optimization': opt, 'clock_mhz': float(m['clk'] or 20),
                'operation': OPS[r['cmd']], 't_exec_ms': t_exec, 't_la_ms': t_la,
                'latency_ms': t_la if not np.isnan(t_la) else t_exec,
            })
    m_dt = re.match(r'exp_(\d{4})(\d{2})(\d{2})_(\d{2})(\d{2})\d{2}_(.+)_\d{4}', e)
    dt_str = f'{m_dt.group(3)}/{m_dt.group(2)} {m_dt.group(4)}:{m_dt.group(5)}' if m_dt else e
    metodo = 'LA 24MHz (PA0) + UART' if has_t_la else ('LA CSV parcial + UART' if la_files else 'UART (host)')
    hist_summary.append({
        'campanha': f'#{idx:02d}', 'experiment': e, 'data_hora': dt_str,
        'configs': len(files), 'algoritmos': ', '.join(sorted(algs_e)),
        'flags': ', '.join(sorted(opts_e)), 'observacoes': n_e, 'instrumentacao': metodo
    })

df_hist = pd.DataFrame(all_rows)
df_hist.to_csv(OUT / 'bench_history_tidy.csv', index=False)
df_campaigns = pd.DataFrame(hist_summary)
df_campaigns.to_csv(OUT / 'campaigns_history.csv', index=False)
print(f'Total acumulado nas {len(bench_exps)} campanhas de bancada: {len(df_hist)} observações')
display(df_campaigns)

In [ ]:
# Evolução histórica das 17 campanhas (volume de coletas e latência Encaps por campanha)
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

fig = plt.figure(figsize=(11.5, 6.2), dpi=300)
gs = gridspec.GridSpec(2, 1, height_ratios=[1.0, 1.35], hspace=0.38)
ax1 = fig.add_subplot(gs[0])
x = np.arange(len(df_campaigns))
obs = df_campaigns['observacoes'].tolist()
colors = ['#0072B2' if 'PA0' in m else ('#E69F00' if 'CSV' in m else '#56B4E9') for m in df_campaigns['instrumentacao']]
bars = ax1.bar(x, obs, color=colors, edgecolor='#222222', linewidth=0.6, width=0.62)
for b, n_val in zip(bars, obs):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 45, f'{n_val}', ha='center', va='bottom', fontsize=7, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f"{r.campanha}\n({r.data_hora.split()[0]})" for _, r in df_campaigns.iterrows()], fontsize=6.8)
ax1.set_ylabel('Ensaios coletados (N)', fontsize=8.5, fontweight='bold')
ax1.set_ylim(0, max(obs) * 1.18)
ax1.set_title('Histórico das 17 Campanhas de Bancada no ATmega4809 (10.124 observações brutas)', fontsize=10, fontweight='bold', pad=8)
ax1.legend(handles=[
    Patch(facecolor='#56B4E9', edgecolor='#222', label='Medição Serial no Host (t_exec_ms)'),
    Patch(facecolor='#E69F00', edgecolor='#222', label='Captura Parcial LA (-O3) + Serial'),
    Patch(facecolor='#0072B2', edgecolor='#222', label='Campanha Final LA 24 MHz no pino PA0 (t_la_ms)'),
], loc='upper left', frameon=True, fontsize=7.5)

ax2 = fig.add_subplot(gs[1])
for alg, col, mk in [('ctru-light', '#D55E00', 's'), ('kyber512', '#0072B2', 'o'), ('ecdh', '#009E73', '^')]:
    sub = df_hist[(df_hist.algorithm == alg) & (df_hist.variant == 'asm') & (df_hist.optimization == '-O3') & (df_hist.operation == 'encaps')]
    g = sub.groupby('campaign_idx')['latency_ms'].agg(['mean', 'std']).reset_index()
    ax2.errorbar(g['campaign_idx'] - 1, g['mean'], yerr=g['std'].fillna(0), fmt=f'-{mk}', color=col,
                 label=f'{LABEL[alg]} (asm -O3, encaps)', markersize=4.5, linewidth=1.2, capsize=2.5)
ax2.set_yscale('log')
ax2.set_xticks(x)
ax2.set_xticklabels([f"{r.campanha}\n{r.data_hora.split()[1]}" for _, r in df_campaigns.iterrows()], fontsize=6.8)
ax2.set_ylabel('Latência Encaps (ms, escala log)', fontsize=8.5, fontweight='bold')
ax2.set_xlabel('Sequência cronológica das campanhas de bancada (04/11 a 08/11)', fontsize=8.5, fontweight='bold')
ax2.set_ylim(40, 12000)
ax2.legend(loc='center left', frameon=True, fontsize=7.5)
fig.savefig(FIG / 'historico_todos_experimentos.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG / 'historico_todos_experimentos.pdf', bbox_inches='tight')
fig.savefig(SLIDE_FIG / 'historico_todos_experimentos.png', dpi=300, bbox_inches='tight')
plt.close(fig)
display(IPImage(filename=str(FIG / "historico_todos_experimentos.png")))

## 2. Isolamento do atraso serial via Analisador Lógico (`PA0`) e base *tidy*

Na campanha #17 (`exp_20261108_155653_bench_campanha_completa_la_nix_8289`), cada execução registrou simultaneamente:
- `t_exec_ms`: o tempo medido no script Python (`trigger.py`) entre o envio do comando na porta serial e o recebimento do primeiro byte da resposta;
- `t_la_ms`: a largura física do pulso em nível alto no pino `PA0` (`D2`), amostrada a 100 kHz ($10\,\mu\text{s}$ de resolução) pelo analisador lógico.

Comparando as duas colunas na mesma execução (tabela abaixo), fica claro o efeito da serialização UART a 115.200 bps: o envio de chaves públicas e cifras adiciona **22,7 a 31,3 ms** fixos, e na operação `msgcrypt` (que é apenas um XOR de 16 bytes com a chave compartilhada) o tempo real de CPU no pino `PA0` é **0,02 ms ($20\,\mu\text{s}$)**, enquanto a leitura via UART marcava ~39,3 ms por conta da troca de mensagens no barramento serial.

Por esse motivo, uso a campanha #17 (`df`, salva em `results/processed/bench_tidy.csv`, com $N=30$ balanceado nas 12 configurações = 1.440 observações) como base principal das tabelas e testes estatísticos de latência pura de CPU.

In [ ]:
LA_EXP = 'exp_20261108_155653_bench_campanha_completa_la_nix_8289'
df = df_hist[df_hist.experiment == LA_EXP].copy()
df['t_exec_ms'] = df['t_la_ms']  # usa a largura do pulso físico no pino PA0 como latência principal
df.to_csv(OUT / 'bench_tidy.csv', index=False)

# Comparação direta entre a largura do pulso no pino PA0 (t_la_ms) e o tempo serial no host
sub_la = df_hist[(df_hist.experiment == LA_EXP) & (df_hist.variant == 'asm') & (df_hist.optimization == '-O3')]
cmp_uart = sub_la.groupby(['algorithm', 'operation']).agg(
    la_gpio_ms=('t_la_ms', 'mean'),
    uart_host_ms=('t_exec_ms', 'mean'),
    payload_bytes=('payload_bytes', 'first')
).reset_index()
cmp_uart['overhead_uart_ms'] = cmp_uart['uart_host_ms'] - cmp_uart['la_gpio_ms']
cmp_uart.round(2).to_csv(OUT / 'la_vs_uart_overhead.csv', index=False)
display(cmp_uart.round(2))

In [ ]:
# Reconstrói o oscilograma da 1ª rodada do Kyber-512 (asm -O3) a partir do CSV bruto do analisador lógico
# e monta a figura composta da bancada (slides/figures/slide_bancada_setup.png).
csv_path = EXP / LA_EXP / 'raw' / 'la_kyber512_asm_O3_20MHz.csv'
times, signals = [], []
with open(csv_path, 'r') as f:
    for row in csv.reader(f):
        if not row or row[0].startswith('#') or row[0] == 'sample_idx': continue
        t_s, d0 = float(row[1]), int(row[2])
        times.append(t_s * 1000.0); signals.append(d0)
        if t_s > 1.35: break
times, signals = np.array(times), np.array(signals)

UFRGS_RED, BLACK, GRAY_TEXT, GRAY_LINE = '#c8102e', '#111827', '#4B5563', '#E5E7EB'
fig, ax = plt.subplots(figsize=(14, 4.2), dpi=300)
ax.step(times, signals, where='post', color=BLACK, linewidth=2.0, zorder=4)
for op in [
    {'name': 'KeyGen', 't0': 114.52, 't1': 353.77, 'dur': '239,3 ms', 'id_pulse': 99.23, 'id_num': '1'},
    {'name': 'Encaps', 't0': 511.05, 't1': 825.60, 'dur': '314,6 ms', 'id_pulse': 488.95, 'id_num': '2'},
    {'name': 'Decaps', 't0': 975.98, 't1': 1263.09, 'dur': '287,1 ms', 'id_pulse': 947.09, 'id_num': '3'},
]:
    t0, t1 = op['t0'], op['t1']; mid = (t0 + t1) / 2
    ax.axvspan(t0, t1, color=UFRGS_RED, alpha=0.08, zorder=2)
    ax.plot([t0, t0], [-0.05, 1.35], color=UFRGS_RED, linestyle=':', linewidth=1.0)
    ax.plot([t1, t1], [-0.05, 1.35], color=UFRGS_RED, linestyle=':', linewidth=1.0)
    ax.annotate('', xy=(t0, 1.30), xytext=(t1, 1.30), arrowprops=dict(arrowstyle='<->', color=UFRGS_RED, lw=1.4))
    ax.text(mid, 1.42, f"{op['name']}: Δt = {op['dur']}", fontsize=10, fontweight='bold', color=UFRGS_RED, ha='center')
    ax.text(op['id_pulse'], 1.15, f"ID ({op['id_num']})", fontsize=7.5, color=GRAY_TEXT, ha='center', style='italic')
ax.set_xlim(40, 1340); ax.set_ylim(-0.25, 1.85); ax.set_yticks([0, 1])
ax.set_yticklabels(['0 (LOW)\n0 V', '1 (HIGH)\n5 V'], fontsize=9, fontweight='bold')
ax.set_ylabel('GPIO PA0 (D2)', fontsize=10, fontweight='bold')
ax.set_xlabel('Tempo (ms) — Captura Física @ 24 MHz', fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_title('Forma de Onda Digital — Janela de Execução Bare-Metal (Kyber-512 asm @ 20 MHz)', fontsize=12, fontweight='bold', loc='left', pad=14)
fig.savefig(FIG / 'oscillogram.png', dpi=300, bbox_inches='tight')
fig.savefig(SLIDE_FIG / 'slide_oscillogram.png', dpi=300, bbox_inches='tight')
plt.close(fig)
display(IPImage(filename=str(FIG / "oscillogram.png")))

# Monta a imagem composta da bancada + oscilograma para os slides
if (SLIDE_FIG / 'bancada.jpg').exists():
    img_b = Image.open(SLIDE_FIG / 'bancada.jpg'); img_o = Image.open(SLIDE_FIG / 'slide_oscillogram.png')
    th = 1000
    wb, wo = int(img_b.width * th / img_b.height), int(img_o.width * th / img_o.height)
    gap, ex_h = 80, 160; tw, tot_h = wb + wo + gap, th + ex_h
    fig2, ax2 = plt.subplots(figsize=(tw / 200, tot_h / 200), dpi=200)
    ax2.set_xlim(0, tw); ax2.set_ylim(0, tot_h); ax2.axis('off')
    ax2.imshow(img_b.resize((wb, th), Image.Resampling.LANCZOS), extent=[0, wb, ex_h, tot_h])
    ax2.imshow(img_o.resize((wo, th), Image.Resampling.LANCZOS), extent=[wb + gap, tw, ex_h, tot_h])
    ax2.text(wb / 2, ex_h / 2 + 15, 'Bancada física: Arduino Nano Every (ATmega4809 @ 20 MHz) + Analisador Lógico (24 MHz)', ha='center', va='center', fontsize=10, fontweight='bold')
    ax2.text(wb + gap + wo / 2, ex_h / 2 + 15, 'Forma de onda real na GPIO PA0: Pulsos de ID e janelas de execução criptográfica pura', ha='center', va='center', fontsize=10, fontweight='bold')
    fig2.savefig(SLIDE_FIG / 'slide_bancada_setup.png', dpi=200, bbox_inches='tight')
    plt.close(fig2)
    display(IPImage(filename=str(SLIDE_FIG / "slide_bancada_setup.png")))

## 3. Sanidade dos dados e controle experimental

Antes de agregar os números, conferi se todas as execuções terminaram com `status = OK`, se o segredo compartilhado do `decaps` bateu com o do `encaps` (`kem_validated = True`) e se a decifragem do `msgcrypt` recuperou a mensagem original (`msg_validated = True`). Também plotei a série temporal das repetições para garantir que não houve deriva térmica ao longo da bateria.

In [ ]:
print('status (campanha LA):', df.status.value_counts().to_dict())
print('kem_validated (decaps):', df.loc[df.operation == 'decaps', 'kem_validated'].value_counts(dropna=False).to_dict())
print('msg_validated (msgcrypt):', df.loc[df.operation == 'msgcrypt', 'msg_validated'].value_counts(dropna=False).to_dict())
df.groupby(['algorithm', 'variant', 'optimization']).size().rename('n_obs').to_frame()

In [ ]:
sub_drift = df[(df.algorithm == 'kyber512') & (df.variant == 'asm') & (df.optimization == '-O3')].copy()
sub_drift['seq'] = sub_drift.groupby('operation').cumcount()
p_drift = (
    p9.ggplot(sub_drift, p9.aes(x='seq', y='t_exec_ms', color='operation'))
    + p9.geom_point(size=1.4, alpha=0.8)
    + p9.labs(x='Ordem de execução (índice dentro da operação)', y='Latência no pino PA0 (ms)',
              title='Kyber-512 asm -O3: verificação de estabilidade temporal (N = 30)')
    + p9.scale_color_brewer(type='qual', palette='Set1', name='Operação')
    + p9.theme_bw(base_size=8)
    + p9.theme(legend_position='top', figure_size=(7, 2.6))
)
fig_drift = p_drift.draw()
fig_drift.savefig(FIG / 'drift_kyber_asm.png', dpi=200, bbox_inches='tight')
fig_drift.savefig(FIG / 'drift_kyber_asm.pdf', bbox_inches='tight')
plt.close(fig_drift)
display(IPImage(filename=str(FIG / "drift_kyber_asm.png")))

## 4. Sumário estatístico (média, IC 95% t-Student, mediana, IQR e pilha)

Calculo abaixo as estatísticas descritivas para cada uma das 48 células (`3 algoritmos × 2 variantes × 2 flags × 4 operações`), salvando em `results/processed/summary_bench.csv` (campanha completa com Analisador Lógico, $N=30$ por célula) e também em `results/processed/summary_history_serial.csv` (média histórica via UART das 17 campanhas).

In [ ]:
def summarise(x):
    n = len(x); m = x.mean(); s = x.std(ddof=1)
    h = stats.t.ppf(0.975, n - 1) * s / np.sqrt(n) if n > 1 else np.nan
    q1, q3 = x.quantile([.25, .75])
    return pd.Series({'n': n, 'mean_ms': m, 'sd_ms': s, 'ci95_ms': h,
                      'median_ms': x.median(), 'iqr_ms': q3 - q1, 'cv_pct': 100 * s / m})

KEYS = ['algorithm', 'variant', 'optimization', 'operation']
summ = df.groupby(KEYS).t_exec_ms.apply(summarise).unstack()
summ['stack_peak_b'] = df.groupby(KEYS).stack_peak.max()
summ['sram_free_pct'] = 100 * (6144 - summ.stack_peak_b) / 6144
summ['energy_uj'] = np.nan  # pendente: calibração do resistor shunt para a etapa final
summ = summ.reset_index()
summ.to_csv(OUT / 'summary_bench.csv', index=False)
display(summ.round(2))

## 5. Ocupação de memória Flash (`.hex` gravados)

O consumo de Flash ROM (`.text` + `.data`) é lido diretamente dos registros de dados (tipo `00`) dos arquivos Intel HEX em `data/experiments/*/bin/*.hex`. Assim a verificação do tamanho dos binários independe de ter o `avr-gcc` instalado na máquina que roda o caderno.

In [ ]:
def hex_bytes(path):
    n = 0
    for line in open(path):
        if line.startswith(':') and line[7:9] == '00':
            n += int(line[1:3], 16)
    return n

HP = re.compile(r'firmware-(?P<alg>kyber512|ctru-light|ecdh)-(?P<variant>asm|ref)-[\d.]+MHz-(?P<opt>O3|Os)\.hex')
hx = []
for f in sorted(glob.glob(str(EXP / 'exp_*' / 'bin' / '*.hex'))):
    m = HP.match(os.path.basename(f))
    if m:
        hx.append({**m.groupdict(), 'flash_b': hex_bytes(f), 'src': Path(f).parts[-3]})
flash = (pd.DataFrame(hx).groupby(['alg', 'variant', 'opt']).last().reset_index()
         .rename(columns={'alg': 'algorithm', 'opt': 'optimization'}))
flash['optimization'] = '-' + flash.optimization
flash['flash_kb'] = (flash.flash_b / 1024.0).round(2)
flash['flash_free_pct'] = (100 * (49152 - flash.flash_b) / 49152).round(2)
flash.to_csv(OUT / 'flash_sizes.csv', index=False)
display(flash)

In [ ]:
fl = flash.copy()
fl['alg_name'] = fl.algorithm.map(LABEL)
fl['cfg'] = [f'{a}\n{v}' for a, v in zip(fl.alg_name, fl.variant)]
order = ['Kyber-512\nasm', 'Kyber-512\nref', 'CTRU-Light\nasm', 'CTRU-Light\nref', 'ECDH\nasm', 'ECDH\nref']
fl['cfg'] = pd.Categorical(fl.cfg, categories=order, ordered=True)
opt_map = {'-O3': 'Otimização -O3', '-Os': 'Otimização -Os'}
fl['opt_label'] = pd.Categorical(fl.optimization.map(opt_map), categories=['Otimização -O3', 'Otimização -Os'], ordered=True)
p_flash = (
    p9.ggplot(fl, p9.aes(x='cfg', y='flash_kb', fill='opt_label'))
    + p9.geom_col(position=p9.position_dodge(width=0.65), width=0.58, color='black', size=0.4)
    + p9.geom_hline(yintercept=48, linetype='dashed', color='#B22222', size=0.8)
    + p9.annotate('text', x=5.4, y=50.2, label='Capacidade do MCU (48 KB)', color='#B22222', size=7.5, fontweight='bold', ha='right', va='bottom')
    + p9.scale_fill_manual(values={'Otimização -O3': '#0072B2', 'Otimização -Os': '#E69F00'}, name='')
    + p9.scale_y_continuous(limits=(0, 56), breaks=[0, 8, 16, 24, 32, 40, 48], expand=(0, 0, 0.05, 0))
    + p9.labs(x='', y='Flash ocupada (KB)')
    + p9.theme_bw(base_size=8)
    + p9.theme(legend_position='top', legend_direction='horizontal', legend_title=p9.element_blank(), figure_size=(TEXTW, 2.3))
)
fig_flash = p_flash.draw()
fig_flash.savefig(FIG / 'flash.png', dpi=300, bbox_inches='tight')
fig_flash.savefig(FIG / 'flash.pdf', bbox_inches='tight')
fig_flash.savefig(SLIDE_FIG / 'slide_flash.png', dpi=300, bbox_inches='tight')
plt.close(fig_flash)
display(IPImage(filename=str(FIG / "flash.png")))

## 6. Latência por configuração e distribuição temporal (CTRU-Light vs. Kyber-512)

Abaixo estão o gráfico facetado de latência nas 12 configurações (escala logarítmica, já que o ECDH leva segundos enquanto Kyber e CTRU-Light levam dezenas/centenas de milissegundos) e o comparativo direto da distribuição empírica do `KeyGen` (`asm -O3`) entre o CTRU-Light e o Kyber-512.

No CTRU-Light, a geração do polinômio secreta exige que ele seja invertível no anel; quando não é, o algoritmo descarta a amostra e sorteia novamente (*rejection sampling*), adicionando saltos discretos de **+10,1 ms (~202.000 ciclos a 20 MHz)**. Já o Kyber-512 opera em tempo constante, formando um único pico estreito em **239,3 ms**.

In [ ]:
kem = df[df.operation != 'msgcrypt'].copy()
s = summ[summ.operation != 'msgcrypt'].copy()
op_map = {'keygen': 'KeyGen', 'encaps': 'Encaps', 'decaps': 'Decaps'}
panel_map = {('asm', '-O3'): 'Assembly (-O3)', ('asm', '-Os'): 'Assembly (-Os)',
             ('ref', '-O3'): 'Referência (-O3)', ('ref', '-Os'): 'Referência (-Os)'}
s['op_label'] = pd.Categorical(s.operation.map(op_map), categories=['KeyGen', 'Encaps', 'Decaps'], ordered=True)
s['alg_label'] = pd.Categorical(s.algorithm.map(LABEL), categories=['Kyber-512', 'CTRU-Light', 'ECDH'], ordered=True)
s['panel'] = pd.Categorical([panel_map[(v, o)] for v, o in zip(s.variant, s.optimization)],
                            categories=['Assembly (-O3)', 'Assembly (-Os)', 'Referência (-O3)', 'Referência (-Os)'], ordered=True)
s['ymin'] = s.mean_ms - s.ci95_ms; s['ymax'] = s.mean_ms + s.ci95_ms
dodge = p9.position_dodge(width=0.35)
p_lat = (
    p9.ggplot(s, p9.aes(x='op_label', y='mean_ms', color='alg_label', group='alg_label'))
    + p9.geom_line(position=dodge, linetype='dashed', size=0.6, alpha=0.85)
    + p9.geom_errorbar(p9.aes(ymin='ymin', ymax='ymax'), position=dodge, width=0.18, size=0.5)
    + p9.geom_point(p9.aes(shape='alg_label'), position=dodge, size=2.2)
    + p9.facet_wrap('~ panel', nrow=1)
    + p9.scale_y_log10(breaks=[100, 1000, 10000], labels=['100 ms', '1 s', '10 s'], limits=(45, 25000))
    + p9.scale_color_manual(values={'Kyber-512': '#0072B2', 'CTRU-Light': '#D55E00', 'ECDH': '#009E73'}, name=' ')
    + p9.scale_shape_manual(values={'Kyber-512': 'o', 'CTRU-Light': 's', 'ECDH': '^'}, name=' ')
    + p9.labs(x='', y='Latência no pino PA0 (escala log)')
    + p9.theme_bw(base_size=8)
    + p9.theme(legend_position='top', legend_direction='horizontal', axis_text_x=p9.element_text(rotation=25, ha='right', size=7), figure_size=(TEXTW, 2.5))
)
fig_lat = p_lat.draw()
fig_lat.savefig(FIG / 'latency.png', dpi=300, bbox_inches='tight')
fig_lat.savefig(FIG / 'latency.pdf', bbox_inches='tight')
plt.close(fig_lat)
display(IPImage(filename=str(FIG / "latency.png")))

# Histograma lado a lado: CTRU-Light (bimodal) vs Kyber-512 (unimodal) em KeyGen asm -O3
ctru_lats = df[(df.algorithm == 'ctru-light') & (df.variant == 'asm') & (df.optimization == '-O3') & (df.operation == 'keygen')]['t_exec_ms'].to_numpy()
kyber_lats = df[(df.algorithm == 'kyber512') & (df.variant == 'asm') & (df.optimization == '-O3') & (df.operation == 'keygen')]['t_exec_ms'].to_numpy()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2), dpi=300)
ax1.hist(ctru_lats, bins=np.arange(59.3, 75.3, 1.0), color='#c8102e', alpha=0.88, edgecolor='#111827', linewidth=1.2, rwidth=0.82)
m1, m2 = ctru_lats[ctru_lats < 66], ctru_lats[ctru_lats >= 66]
ax1.text(m1.mean(), len(m1) + 1.0, f'1ª Tentativa\n{m1.mean():.1f} ms (n = {len(m1)})', ha='center', va='bottom', fontsize=9.5, fontweight='bold')
ax1.text(m2.mean(), len(m2) + 1.0, f'Com Rejeição\n{m2.mean():.1f} ms (n = {len(m2)})', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#c8102e')
ax1.annotate('', xy=(m1.mean() + 0.55, 8.5), xytext=(m2.mean() - 0.55, 8.5), arrowprops=dict(arrowstyle='<->', color='#111827', lw=1.8))
ax1.text((m1.mean() + m2.mean()) / 2, 9.4, f'+{m2.mean() - m1.mean():.1f} ms', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_xlim(58.5, 75.5); ax1.set_ylim(0, 34); ax1.set_xlabel('Tempo de Execução (ms)', fontweight='bold'); ax1.set_ylabel('Número de Execuções (N = 30)', fontweight='bold')
ax1.set_title('CTRU-Light KeyGen — Bimodal', fontsize=11.5, fontweight='bold')

ax2.hist(kyber_lats, bins=np.arange(231.7, 248.7, 1.0), color='#0072B2', alpha=0.88, edgecolor='#111827', linewidth=1.2, rwidth=0.82)
k_main = kyber_lats[kyber_lats < 242]; med_k = np.median(kyber_lats)
ax2.text(med_k, len(k_main) + 1.0, f'Pico Único (Tempo Constante)\n{med_k:.1f} ms (n = {len(k_main)}/30)', ha='center', va='bottom', fontsize=9.5, fontweight='bold', color='#0072B2')
ax2.set_xlim(231.0, 248.0); ax2.set_ylim(0, 34); ax2.set_xlabel('Tempo de Execução (ms)', fontweight='bold')
ax2.set_title('Kyber-512 KeyGen — Unimodal', fontsize=11.5, fontweight='bold')
for ax in (ax1, ax2): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(w_pad=2.5)
fig.savefig(FIG / 'bimodality.png', dpi=300, bbox_inches='tight')
fig.savefig(FIG / 'bimodality.pdf', bbox_inches='tight')
fig.savefig(SLIDE_FIG / 'slide_bimodality.png', dpi=300, bbox_inches='tight')
plt.close(fig)
display(IPImage(filename=str(FIG / "bimodality.png")))

## 7. Comparação entre alternativas (razão de médias e IC 95% via Welch em escala log)

Para quantificar os *speedups* (como `ref` vs. `asm`, `-Os` vs. `-O3` e as razões entre algoritmos) com intervalo de confiança de 95%, aplico o teste t de Welch sobre $\log(T)$ e exponencio a diferença e seus limites.

In [ ]:
def compare(a, b):
    la, lb = np.log(a), np.log(b)
    d = lb.mean() - la.mean()
    se = np.sqrt(la.var(ddof=1) / len(la) + lb.var(ddof=1) / len(lb))
    dof = se**4 / ((la.var(ddof=1) / len(la))**2 / (len(la) - 1) + (lb.var(ddof=1) / len(lb))**2 / (len(lb) - 1))
    h = stats.t.ppf(0.975, dof) * se
    p = stats.ttest_ind(lb, la, equal_var=False).pvalue
    return np.exp(d), np.exp(d - h), np.exp(d + h), p

def table(factor, base, alt):
    out = []
    for keys, grp in kem.groupby([k for k in KEYS if k != factor]):
        a = grp.loc[grp[factor] == base, 't_exec_ms']; b = grp.loc[grp[factor] == alt, 't_exec_ms']
        if len(a) > 1 and len(b) > 1:
            r, lo, hi, p = compare(a, b)
            out.append({**dict(zip([k for k in KEYS if k != factor], keys)),
                        f'razao_{alt}/{base}': r, 'ic_lo': lo, 'ic_hi': hi, 'p_welch': p})
    return pd.DataFrame(out)

for name, f, b, a in [('ref/asm', 'variant', 'asm', 'ref'), ('-Os/-O3', 'optimization', '-O3', '-Os'),
                      ('kyber/ctru', 'algorithm', 'ctru-light', 'kyber512'),
                      ('ecdh/kyber', 'algorithm', 'kyber512', 'ecdh'),
                      ('ecdh/ctru', 'algorithm', 'ctru-light', 'ecdh')]:
    t = table(f, b, a)
    t.to_csv(OUT / f"cmp_{name.replace('/', '_vs_').replace('-', '')}.csv", index=False)
    print('\n==', name); display(t.round(4))

## 8. Modelo linear fatorial (ANOVA sobre $\log T$)

Ajusto o modelo fatorial completo $\log(T) = \mu + \text{alg} + \text{variante} + \text{opt} + \text{op} + \text{interações} + \varepsilon$ sobre as 1.080 observações de KEM ($3 \times 2 \times 2 \times 3 \times 30$) para separar a fração da variância total ($\eta^2$) explicada por cada fator.

In [ ]:
kem['logT'] = np.log(kem.t_exec_ms)
fit = smf.ols('logT ~ C(algorithm) * C(variant) * C(optimization) * C(operation)', data=kem).fit()
aov = sm.stats.anova_lm(fit, typ=2)
aov['eta2'] = aov.sum_sq / aov.sum_sq.sum()
aov = aov.sort_values('eta2', ascending=False)
aov.to_csv(OUT / 'anova_latency.csv')
print(f'R² = {fit.rsquared:.5f}   n = {int(fit.nobs)}')
display(aov.round(5))

fig, axs = plt.subplots(1, 2, figsize=(7, 2.6))
axs[0].scatter(fit.fittedvalues, fit.resid, s=3, alpha=.4); axs[0].axhline(0, c='k', lw=.8)
axs[0].set_xlabel('ajustado (log ms)'); axs[0].set_ylabel('resíduo')
sm.qqplot(fit.resid, line='s', ax=axs[1], markersize=2); axs[1].set_title('')
fig.savefig(FIG / 'residuals.pdf'); fig.savefig(FIG / 'residuals.png', dpi=200)
plt.close(fig)
display(IPImage(filename=str(FIG / "residuals.png")))

## 9. Trade-off memória SRAM × latência e exportação das tabelas LaTeX

Gero abaixo o gráfico de *trade-off* entre o pico de pilha na SRAM (medido por *stack watermarking* com `0xAA`) e a latência no pino `PA0`, além das duas tabelas em LaTeX (`results/tables/results.tex` e `results/tables/anova.tex`).

In [ ]:
s = summ[summ.operation != 'msgcrypt'].copy()
s['cfg_label'] = [f'{LABEL[a]} ({v})' for a, v in zip(s.algorithm, s.variant)]
cfg_order = ['Kyber-512 (asm)', 'Kyber-512 (ref)', 'CTRU-Light (asm)', 'CTRU-Light (ref)', 'ECDH (asm)', 'ECDH (ref)']
s['cfg_label'] = pd.Categorical(s.cfg_label, categories=cfg_order, ordered=True)
s['ymin'] = s.mean_ms - s.ci95_ms; s['ymax'] = s.mean_ms + s.ci95_ms
colors = {'Kyber-512 (asm)': '#0072B2', 'Kyber-512 (ref)': '#0072B2', 'CTRU-Light (asm)': '#D55E00', 'CTRU-Light (ref)': '#D55E00', 'ECDH (asm)': '#009E73', 'ECDH (ref)': '#009E73'}
fills = {'Kyber-512 (asm)': '#0072B2', 'Kyber-512 (ref)': 'white', 'CTRU-Light (asm)': '#D55E00', 'CTRU-Light (ref)': 'white', 'ECDH (asm)': '#009E73', 'ECDH (ref)': 'white'}
shapes = {'Kyber-512 (asm)': 'o', 'Kyber-512 (ref)': 's', 'CTRU-Light (asm)': 'o', 'CTRU-Light (ref)': 's', 'ECDH (asm)': 'o', 'ECDH (ref)': 's'}
p_trade = (
    p9.ggplot(s, p9.aes(x='stack_peak_b', y='mean_ms', color='cfg_label', fill='cfg_label', shape='cfg_label'))
    + p9.geom_errorbar(p9.aes(ymin='ymin', ymax='ymax'), width=25, size=0.5, alpha=0.8)
    + p9.geom_point(size=2.8, stroke=1.1)
    + p9.scale_x_continuous(limits=(350, 3500), breaks=[500, 1000, 1500, 2000, 2500, 3000], labels=['500 B', '1.000 B', '1.500 B', '2.000 B', '2.500 B', '3.000 B'])
    + p9.scale_y_log10(limits=(45, 25000), breaks=[100, 1000, 10000], labels=['100 ms', '1 s', '10 s'])
    + p9.scale_color_manual(values=colors, name='Configuração') + p9.scale_fill_manual(values=fills, name='Configuração') + p9.scale_shape_manual(values=shapes, name='Configuração')
    + p9.guides(color=p9.guide_legend(ncol=2), fill=p9.guide_legend(ncol=2), shape=p9.guide_legend(ncol=2))
    + p9.labs(x='Pico de pilha (bytes)', y='Latência média (escala log)')
    + p9.theme_bw(base_size=8) + p9.theme(legend_position=(0.77, 0.77), legend_title=p9.element_blank(), figure_size=(TEXTW, 2.5))
)
fig_trade = p_trade.draw()
ax_tr = fig_trade.axes[0]; ax_top = ax_tr.twiny(); ax_top.set_xlim(ax_tr.get_xlim())
top_pcts = [10, 20, 30, 40, 50]
ax_top.set_xticks([6144 * p / 100 for p in top_pcts]); ax_top.set_xticklabels([f'{p}%' for p in top_pcts], fontsize=7)
ax_top.set_xlabel('Ocupação da SRAM do ATmega4809 (total 6 KB)', fontsize=7.5, labelpad=4); ax_top.grid(False)
fig_trade.savefig(FIG / 'tradeoff.png', dpi=300, bbox_inches='tight')
fig_trade.savefig(FIG / 'tradeoff.pdf', bbox_inches='tight')
fig_trade.savefig(SLIDE_FIG / 'slide_tradeoff.png', dpi=300, bbox_inches='tight')
plt.close(fig_trade)
display(IPImage(filename=str(FIG / "tradeoff.png")))

# Exportação das tabelas LaTeX (results.tex e anova.tex)
t = summ[summ.operation != 'msgcrypt'].set_index(['algorithm', 'variant', 'optimization', 'operation'])
fl = flash.set_index(['algorithm', 'variant', 'optimization']).flash_b
lines = [r'\begin{tabular}{@{}lllrrrrrr@{}}', r'\toprule',
         r' & & & \multicolumn{3}{c}{Latência (ms), média $\pm$ IC 95\%} & & & \\',
         r'\cmidrule(lr){4-6}',
         r'Algoritmo & Var. & Otim. & KeyGen & Encaps & Decaps & Pilha (B) & Flash (KB) & $N$ \\', r'\midrule']
for k, alg in enumerate(['kyber512', 'ctru-light', 'ecdh']):
    if k: lines.append(r'\addlinespace')
    first = True
    for var in ['asm', 'ref']:
        for opt in ['-O3', '-Os']:
            r = t.loc[(alg, var, opt)]
            cells_str = [f"{br(r.loc[o, 'mean_ms'])} $\\pm$ {br(r.loc[o, 'ci95_ms'])}" for o in ['keygen', 'encaps', 'decaps']]
            lines.append(' & '.join([LABEL[alg] if first else '', var, f'$-${opt[1:]}', *cells_str,
                                     f'{int(r.stack_peak_b.max()):,}'.replace(',', '.'),
                                     br(fl.loc[(alg, var, opt)] / 1024), str(int(r.n.min()))]) + r' \\')
            first = False
lines += [r'\bottomrule', r'\end{tabular}']
(TAB / 'results.tex').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print((TAB / 'results.tex').read_text(encoding='utf-8'))
tnames = {
    "C(algorithm)": "Algoritmo", "C(variant)": "Implementação", "C(optimization)": "Otimização", "C(operation)": "Operação",
    "C(algorithm):C(operation)": r"Algoritmo $\times$ Operação",
    "C(algorithm):C(variant)": r"Algoritmo $\times$ Implementação",
    "C(algorithm):C(variant):C(operation)": r"Algoritmo $\times$ Implementação $\times$ Operação",
}
alines = [r"\begin{tabular}{@{}lrrr@{}}", r"\toprule", r"Termo & GL & $F$ & $\eta^2$ (\%) \\", r"\midrule"]
for idx_row, r in aov.drop("Residual").iterrows():
    if r.eta2 * 100 >= 0.02 and idx_row in tnames:
        alines.append(f"{tnames[idx_row]} & {int(r.df)} & {int(round(r.F)):,} & {br(r.eta2 * 100, 2)} \\".replace(",", "."))
res = aov.loc["Residual"]
alines += [r"\midrule", f"Resíduo & {int(res.df)} & -- & {br(res.eta2 * 100, 2)} \\", r"\bottomrule", r"\end{tabular}"]
(TAB / "anova.tex").write_text("\n".join(alines) + "\n", encoding="utf-8")
print((TAB / 'anova.tex').read_text(encoding='utf-8'))


## 10. O que os dados mostram até aqui

1. **Isolamento do observador (UART vs. pino `PA0`):** medir pelo pino físico ligado ao analisador lógico removeu de 22,7 a 31,3 ms de atraso serial nas operações do KEM e mostrou que o `msgcrypt` dura apenas 0,02 ms de CPU real (todo o resto dos ~39 ms medidos no host vinha da fila serial USB/CDC).
2. **Fatores que dominam o tempo de CPU:** na ANOVA sobre $\log(T)$, a escolha do algoritmo explica **92,92%** da variação total, a variante (`asm` vs. `ref`) explica **4,54%** e a interação `algoritmo × operação` explica **2,02%**. A flag do compilador (`-Os` vs. `-O3`) responde por menos de **0,05%** da variação de tempo.
3. **Reticulados vs. Curvas Elípticas em 8 bits:** no ATmega4809 (multiplicador de hardware de apenas $8 \times 8$ bits), o ECDH secp256r1 (`asm -O3`) leva **3,18 s** no `KeyGen`/`Decaps` e **6,36 s** no `Encaps`, enquanto o Kyber-512 leva **239 a 315 ms** ($11\times$ a $20\times$ mais rápido) e o CTRU-Light leva **61 a 120 ms** ($26\times$ a $104\times$ mais rápido). A aritmética polinomial com coeficientes curtos adapta-se muito melhor à ALU de 8 bits do que a aritmética multi-precisão de 256 bits.
4. **O gargalo real é a Flash de 48 KB (onde `-Os` e `asm` são obrigatórios):** em C de referência com `-O3`, o Kyber-512 ocupa **44,5 KB** (93% da Flash) e o CTRU-Light ocupa **47,0 KB** (98% da Flash), praticamente inviabilizando colocar a aplicação IoT junto no mesmo microcontrolador. Passando para a versão `asm` com `-Os`, o Kyber-512 cai para **13,9 KB** (71% de Flash livre) e o CTRU-Light para **21,3 KB** (56% livre), perdendo apenas 1% a 7% em tempo de execução.
5. **Trade-off Kyber-512 vs. CTRU-Light na SRAM:** o CTRU-Light é mais rápido, mas exige até **3.174 B** de pilha no `decaps` (~52% dos 6 KB de SRAM), enquanto o Kyber-512 usa no máximo **1.029 B** (~17% da SRAM, deixando 83% livre).
6. **Bimodalidade do CTRU-Light:** enquanto o Kyber-512 e o ECDH têm tempo constante (CV < 0,5%), o `KeyGen` e o `Encaps` do CTRU-Light exibem dois picos claros (~61,8 ms na 1ª tentativa e ~71,9 ms quando há reamostragem por rejeição do polinômio).
7. **Próximo passo para o relatório final:** integrar o resistor *shunt* na linha de alimentação do ATmega4809 para medir a energia em microjoules ($\mu\text{J}$) sincronizada pela janela do pino `PA0` e completar as medições em 10 MHz e 5 MHz.